In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import StratifiedKFold

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering import (
    prepare_features,
    add_title_hierarchy_features,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

TARGET_COLUMN = "Цена"

train = pd.read_parquet(
    PROCESSED_DIR / "train_canonical.parquet"
)

y = train[TARGET_COLUMN].copy()

X_v6 = add_title_hierarchy_features(
    prepare_features(
        train.drop(columns=[TARGET_COLUMN])
    )
)

RAW_DUPLICATE_NUMERIC_COLUMNS = [
    "Пробег",
    "Расход",
    "Количество цилиндров",
    "Двери",
    "Количество кресел",
]

EXCLUDED_COLUMNS = [
    "car_id",
    "Предложение",
] + RAW_DUPLICATE_NUMERIC_COLUMNS

# Ключевое отличие v6 от v5:
# исключаем raw «Полное название».
EXCLUDED_COLUMNS_V6 = EXCLUDED_COLUMNS + [
    "Полное название",
]

base_numeric_columns = [
    "Год выпуска",
    "Оценка эксперта",
    "Количество владельцев",
    "Пробег_число",
    "Расход_л_на_100км",
    "Двигатель_цилиндры",
    "Двигатель_объём_л",
    "Двери_число",
    "Кресла_число",
]

title_numeric_columns_v4 = [
    "Название_число_слов",
    "Название_есть_4X4",
    "Название_есть_AWD",
    "Название_есть_TURBO",
    "Название_есть_SPORT",
    "Название_есть_HYBRID",
    "Название_есть_GT",
    "Название_есть_LUXURY",
    "Название_есть_DIESEL",
]

title_hierarchy_numeric_columns = [
    "Название_мощность_kw",
    "Название_есть_мощность_kw",
    "Название_есть_AMG",
    "Название_есть_M_SPORT",
    "Название_есть_RS",
    "Название_есть_S_LINE",
    "Название_есть_GTI",
    "Название_есть_HSE",
    "Название_есть_SR5",
    "Название_есть_GXL",
    "Название_есть_LIMITED",
    "Название_есть_PREMIUM",
    "Название_есть_COMFORTLINE",
    "Название_есть_ASCENT",
    "Название_есть_ACTIVE",
    "Название_есть_ELITE",
    "Название_есть_TDI",
    "Название_есть_TSI",
    "Название_есть_TFSI",
    "Название_есть_CDI",
    "Название_есть_V6",
    "Название_есть_V8",
]

numeric_columns_v6 = (
    base_numeric_columns
    + title_numeric_columns_v4
    + title_hierarchy_numeric_columns
)

feature_columns_v6 = [
    column
    for column in X_v6.columns
    if column not in EXCLUDED_COLUMNS_V6
]

categorical_columns_v6 = [
    column
    for column in feature_columns_v6
    if column not in numeric_columns_v6
]

X_model_v6 = X_v6[
    feature_columns_v6
].copy()

for column in categorical_columns_v6:
    X_model_v6[column] = (
        X_model_v6[column]
        .fillna("__MISSING__")
        .astype(str)
    )

assert "Полное название" not in X_model_v6.columns
assert X_model_v6.shape[1] == 58

cv_target_bins = pd.qcut(
    y,
    q=10,
    labels=False,
    duplicates="drop",
).to_numpy()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

print("X_model_v6:", X_model_v6.shape)
print("Numeric:", len(numeric_columns_v6))
print("Categorical:", len(categorical_columns_v6))

X_model_v6: (8340, 58)
Numeric: 40
Categorical: 18


In [2]:
def mape_percent(y_true, y_pred) -> float:
    return mean_absolute_percentage_error(y_true, y_pred) * 100


oof_v6_predictions_clean = np.zeros(len(y))
v6_fold_results_clean = []

for fold, (train_fold_idx, valid_fold_idx) in enumerate(
    cv.split(X_model_v6, cv_target_bins),
    start=1,
):
    print(f"\n{'=' * 60}")
    print(f"CatBoost v6 fold {fold}/5")
    print(f"{'=' * 60}")

    X_train_fold = X_model_v6.iloc[train_fold_idx].copy()
    X_valid_fold = X_model_v6.iloc[valid_fold_idx].copy()

    y_train_fold = y.iloc[train_fold_idx].copy()
    y_valid_fold = y.iloc[valid_fold_idx].copy()

    model_fold = CatBoostRegressor(
        loss_function="RMSE",
        iterations=3000,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=5,
        random_seed=42,
        verbose=500,
        allow_writing_files=False,
    )

    model_fold.fit(
        X_train_fold,
        np.log1p(y_train_fold),
        cat_features=categorical_columns_v6,
        eval_set=(
            X_valid_fold,
            np.log1p(y_valid_fold),
        ),
        use_best_model=True,
        early_stopping_rounds=200,
    )

    # Страховка: raw title точно не попал в модель.
    assert "Полное название" not in model_fold.feature_names_

    fold_predictions = np.maximum(
        np.expm1(
            model_fold.predict(X_valid_fold)
        ),
        1,
    )

    oof_v6_predictions_clean[
        valid_fold_idx
    ] = fold_predictions

    fold_mape = mape_percent(
        y_valid_fold,
        fold_predictions,
    )

    v6_fold_results_clean.append(
        {
            "fold": fold,
            "best_iteration": model_fold.get_best_iteration(),
            "validation_mape_pct": fold_mape,
        }
    )

    print(
        f"Fold {fold} MAPE: {fold_mape:.3f}% | "
        f"best iteration: {model_fold.get_best_iteration()}"
    )


CatBoost v6 fold 1/5
0:	learn: 0.6535614	test: 0.6455360	best: 0.6455360 (0)	total: 347ms	remaining: 17m 21s
500:	learn: 0.1509100	test: 0.2110765	best: 0.2110765 (500)	total: 40.1s	remaining: 3m 20s
1000:	learn: 0.1129219	test: 0.2023204	best: 0.2022962 (996)	total: 1m 20s	remaining: 2m 41s
1500:	learn: 0.0909581	test: 0.2000329	best: 0.2000020 (1498)	total: 3m 11s	remaining: 3m 11s
2000:	learn: 0.0746943	test: 0.1988842	best: 0.1988631 (1989)	total: 5m 19s	remaining: 2m 39s
2500:	learn: 0.0626117	test: 0.1982288	best: 0.1982203 (2480)	total: 7m 26s	remaining: 1m 29s
2999:	learn: 0.0532026	test: 0.1979599	best: 0.1979483 (2972)	total: 9m 33s	remaining: 0us

bestTest = 0.1979483271
bestIteration = 2972

Shrink model to first 2973 iterations.
Fold 1 MAPE: 13.413% | best iteration: 2972

CatBoost v6 fold 2/5
0:	learn: 0.6506029	test: 0.6565337	best: 0.6565337 (0)	total: 223ms	remaining: 11m 7s
500:	learn: 0.1601411	test: 0.1933784	best: 0.1933657 (499)	total: 1m 58s	remaining: 9m 49s
10

In [3]:
v6_fold_results_clean_df = pd.DataFrame(
    v6_fold_results_clean
)

v6_oof_mape_clean = mape_percent(
    y,
    oof_v6_predictions_clean,
)

display(v6_fold_results_clean_df)

print(
    f"Fresh v6 OOF MAPE: {v6_oof_mape_clean:.3f}%"
)

v6_oof_clean = pd.DataFrame(
    {
        "car_id": train["car_id"].to_numpy(),
        "y_true": y.to_numpy(),
        "title_v6_pred": oof_v6_predictions_clean,
    }
)

v6_clean_path = (
    REPORTS_DIR
    / "catboost_title_hierarchy_v6_oof_clean.parquet"
)

v6_oof_clean.to_parquet(
    v6_clean_path,
    index=False,
)

print("Saved:", v6_clean_path)

,fold,best_iteration,validation_mape_pct
0,1,2972,13.412829
1,2,2844,12.291362
2,3,2441,12.806219
3,4,2979,12.834264
4,5,2997,12.390038


Fresh v6 OOF MAPE: 12.747%
Saved: C:\temp\shift_ml\reports\catboost_title_hierarchy_v6_oof_clean.parquet


In [4]:
v5_oof_fresh = pd.read_parquet(
    REPORTS_DIR / "catboost_title_hierarchy_v5_oof_fresh.parquet"
)

check = (
    v5_oof_fresh[
        ["car_id", "title_v5_pred"]
    ]
    .merge(
        v6_oof_clean[
            ["car_id", "title_v6_pred"]
        ],
        on="car_id",
        validate="one_to_one",
    )
)

abs_diff = (
    check["title_v5_pred"]
    - check["title_v6_pred"]
).abs()

print("Equal exactly:", abs_diff.eq(0).all())
print("Mean abs difference:", abs_diff.mean())
print("Max abs difference:", abs_diff.max())

assert not abs_diff.eq(0).all()

Equal exactly: False
Mean abs difference: 1228.5564916557191
Max abs difference: 37675.746639531484
